## Plotting stacked absorption profiles compared to Lyman-α and Emission Lines

In [ ]:
from tangelo import spectroscopy as tgspec
from tangelo import constants as tgconst
from tangelo import catalogue_operations as tgcat
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import astropy.table as aptb
from matplotlib import gridspec

In [ ]:
# Load megatable of Lyman alpha sources
# Which spectra source to use? R21 or APER
SPEC_SOURCE = "R21"  #  Where are the spectra from? 'R21' for Richard+21, 'APER' for apertures
SPEC_TYPE   = 'weight_skysub' if SPEC_SOURCE == "R21" else '2fwhm'

tabfile = f"../megatables/fit_catalog_qc_cpts_stack_zeldamcmc_refit_sysv_absv_{SPEC_TYPE}.fits"
tabhdul = fits.open(tabfile)
megatable = aptb.Table(tabhdul[1].data)

# Filter to only include LAEs
lya_mask = (megatable['SNRR'] >= 5) | (megatable['SNRB'] >= 5)
megatable = megatable[lya_mask]

In [ ]:
# Generate a mask for sources with absorption at a specified significance level
sig_threshold = 3  # Example: 3-sigma significance
has_absorption = (
    (-megatable['EW_LI_ABS'] / megatable['EW_LI_ABS_ERR'] >= sig_threshold) |
    (-megatable['EW_HI_ABS'] / megatable['EW_HI_ABS_ERR'] >= sig_threshold) |
    (-megatable['EW_TOT_ABS'] / megatable['EW_TOT_ABS_ERR'] >= sig_threshold)
)

num_absorbers = np.sum(has_absorption)
print(f"Total number of absorbers: {num_absorbers}")

# How many of these also have systemic velocity measurements?
has_sysvel = (megatable['VFLUX_SYS'] / megatable['VFLUX_SYS_ERR'] >= sig_threshold)

num_absorbers_with_sysvel = np.sum(has_sysvel & has_absorption)
print(f"Number of absorbers with systemic velocity measurements: {num_absorbers_with_sysvel}")

In [ ]:

# Generate a figure containing all absorbers with systemic velocity measurements

# These are the lines that, if detected at >3 sigma, can be used to make the stack
good_lines =  ['CIII1907', 'CIII1909', 'HeII1640', 
               'NIII1750', 'NIV1483', 'NIV1487',
               'OIII1660', 'OIII1666', 'SiIII1883', 'SiIII1892']

# These are the default optically thin lines to use if no good lines are detected
optthin_lines = ['HeII1640','OIII1666','CIII1907','CIII1909']

# Absorption line lists — must match step08 definitions exactly
hi_abslines = ['SiIV1394', 'SiIV1403']

# Spectral extraction parameters — must match step08
vel_bin = 75
spec_half_width = 2000

# First figure out dimensions: name how many columns we want, and calculate rows
num_cols = 5
num_rows = int(np.ceil(num_absorbers_with_sysvel / num_cols))
# How many sub-subplots for each subplot?
num_subs = 4 # Lyman alpha, emission lines, low-ion absorption, high-ion absorption

fig = plt.figure(figsize=(4*num_cols,5*num_rows), facecolor='white')
gs  = gridspec.GridSpec(num_rows, num_cols, wspace=0.1, hspace=0.15, figure=fig)

for i, row in enumerate(megatable[has_sysvel & has_absorption]):
    # Determine which subplot we're in
    row_idx = i // num_cols
    col_idx = i % num_cols
    sub_gs = gridspec.GridSpecFromSubplotSpec(4, 1, subplot_spec=gs[row_idx, col_idx], wspace=0, hspace=0)

    # Common avg_lines kwargs — matches step08 exactly
    avg_kwargs = dict(
        absorption=False,
        velbounds=[-spec_half_width, spec_half_width],
        z=row['LPEAKR'] / 1215.67 - 1,
        spec_source=SPEC_SOURCE,
        spec_type=SPEC_TYPE,
        velstep=vel_bin,
    )

    # Plot Lyman alpha (shifted to systemic)
    ax0 = fig.add_subplot(sub_gs[0])
    lya_wl, lya_spec, lya_err = tgspec.avg_lines(row, ['LYALPHA'], **avg_kwargs)
    lya_wl_shifted = lya_wl + row['DELTAV_LYA']  # Shift to systemic velocity

    ax0.plot(lya_wl_shifted, lya_spec, color='slateblue', drawstyle='steps-mid', label=r"Ly$\alpha$")
    ax0.fill_between(lya_wl_shifted, lya_spec - lya_err, lya_spec + lya_err, color='slateblue', alpha=0.3,
                     step='mid', linestyle='')
    ax0.axvline(row['DELTAV_LYA'], color='red', linestyle='--', alpha=0.7)
    # calculate blue peak velocity relative to systemic
    if not np.isnan(row['LPEAKB']):
        dl_blue = (row['LPEAKB'] - row['LPEAKR']) / (1 + row['z'])
        v_blue = (dl_blue / tgconst.wavedict['LYALPHA']) * tgconst.c + row['DELTAV_LYA']
        ax0.axvline(v_blue, color='blue', linestyle='--', alpha=0.7)

    ax0.tick_params(labelbottom=False)  # Hide x-tick labels on top row
    ax0.set_xlim(-1600, 1600)
    # Only put the legend on the first subplot
    if i == 0:
        ax0.legend(frameon=True, facecolor='white', edgecolor='none', framealpha=0.8)

    # Calculate the true redshift using Lyman alpha peak redshift and velocity offset from systemic
    c_kms = tgconst.c
    true_z = ((row['LPEAKR'] / tgconst.wavedict['LYALPHA']) * (1 - row['DELTAV_LYA'] / c_kms)) - 1
    
    # Add object ID text in upper left corner
    ax0.text(0.02, 0.90, f"{row['CLUSTER']} {row['iden']}\n($z={true_z:.3f}$)", 
             transform=ax0.transAxes, fontsize=10, fontweight='bold',
             verticalalignment='top', horizontalalignment='left',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='none'))

    # Use the same emission line selection logic as step07_measuring_systemic_redshifts:
    # call is_true_emitter to check whether >=2 good lines are detected at >=3 sigma;
    # if so, use only those detected lines; otherwise fall back to the default opt-thin list.
    tentem = tgcat.is_true_emitter(row, {x: tgconst.wavedict[x] for x in good_lines}, n=2, sig=3, return_lines=True)
    if tentem[0]:
        em_lines = list(set(tentem[1][0]) & set(good_lines))
        if len(em_lines) == 0:
            print(f"Verified emitter but no usable lines for {row['iden']}, reverting to defaults.")
            em_lines = optthin_lines
        else:
            print(f"Using emission lines for object ID {row['iden']}: {em_lines}")
    else:
        print(f"Insufficient emission lines for {row['iden']}. Using default line list.")
        em_lines = optthin_lines

    ax1 = fig.add_subplot(sub_gs[1], sharex=ax0)
    emis_wl, emis_spec, emis_err = tgspec.avg_lines(row, em_lines, **avg_kwargs)
    emis_wl_shifted = emis_wl + row['DELTAV_LYA']  # Shift to systemic velocity

    ax1.plot(emis_wl_shifted, emis_spec, color='firebrick', drawstyle='steps-mid', label='Opt. thin emission')
    ax1.fill_between(emis_wl_shifted, emis_spec - emis_err, emis_spec + emis_err, color='firebrick', alpha=0.3,
                      step='mid', linestyle='')
    ax1.axvline(0., color='red', linestyle='--', alpha=0.7)
    ax1.tick_params(labelbottom=False)  # Hide x-tick labels on second row
    # Only put the legend and axis label on the first subplot
    if i == 0:
        ax1.set_ylabel('$f_v$ [erg\,s$^{-1}$\,cm$^{-2}$\,(km\,s$^{-1}$)$^{-1}$]')
        ax1.legend(loc='upper right', frameon=True, facecolor='white', edgecolor='none', framealpha=0.8)

    # Determine which line combinations were used to detect absorption.
    # For HI, apply the same flag-filtering as step08 to reproduce the exact stack used during fitting.
    hilines_used = [l for l in hi_abslines if not row[f'FLAG_{l}']]
    hi_line_str = '+'.join(hilines_used) if hilines_used else '+'.join(hi_abslines)

    raw_detected = {}
    for line_type in ['LI', 'HI', 'TOT']:
        if line_type == 'HI' and not np.isnan(row['DV_HI_ABS']):
            raw_detected['HI'] = hi_line_str
            print(f"Object ID {row['iden']} has HI absorption from: {raw_detected['HI']}")
        elif line_type in ['LI', 'TOT'] and not np.isnan(row[f'DV_{line_type}_ABS']):
            dv_lines_key = f'DV_{line_type}_ABS_LINES'
            raw_detected[line_type] = row[dv_lines_key]
            print(f"Object ID {row['iden']} has {line_type} absorption from: {row[dv_lines_key]}")

    # Always show LI (panel 3) and either HI or TOT (panel 4).
    # If only TOT was detected (neither LI nor HI), show LI + TOT.
    # Otherwise show LI + HI (greyed out below if SiIV is out of MUSE range).
    detected_types = {}
    detected_types['LI'] = raw_detected.get('LI', 'SiII1260+CII1334')
    if 'TOT' in raw_detected and 'LI' not in raw_detected and 'HI' not in raw_detected:
        detected_types['TOT'] = raw_detected['TOT']
    else:
        detected_types['HI'] = raw_detected.get('HI', hi_line_str)

    # Generate axes for absorption lines
    absaxes = {abstype: fig.add_subplot(sub_gs[2 + idx], sharex=ax0) for idx, abstype in enumerate(detected_types.keys())}
    typecolors = {'LI': 'teal', 'HI': 'darkgoldenrod', 'TOT': 'indigo'}

    for idx, (abstype, lines) in enumerate(detected_types.items()):

        is_last = (idx == len(detected_types) - 1)
        ax = absaxes[abstype]

        # Generate list of lines from string, filtering out any empty strings
        line_list = [l for l in lines.split('+') if l]
        if not line_list:
            print(f"No valid lines for {abstype} absorption in {row['iden']}, skipping panel.")
            continue

        # Stack the absorption lines using the same method as step08
        wavelength, flux, flux_err = tgspec.avg_lines(row, line_list, **avg_kwargs)

        # Grey out HI panel if SiIV lines are outside MUSE wavelength coverage (all-NaN result)
        if abstype == 'HI' and np.all(np.isnan(flux)):
            ax.set_facecolor('#dddddd')
            ax.text(0.5, 0.5, 'SiIV outside MUSE range\n(redshift too high)',
                    transform=ax.transAxes, ha='center', va='center',
                    fontsize=9, color='#444444', style='italic')
            if is_last:
                ax.set_xlabel('Velocity relative to systemic (km/s)')
                ax.tick_params(labelbottom=True)
            else:
                ax.tick_params(labelbottom=False)
            continue

        # Shift wavelength to systemic velocity
        wavelength_shifted = wavelength + row['DELTAV_LYA']

        # Plot the absorption
        plot_label = '\n'.join(line_list)
        ax.plot(wavelength_shifted, flux, color=typecolors[abstype], label=plot_label,
                 drawstyle='steps-mid')
        ax.fill_between(wavelength_shifted, flux - flux_err, flux + flux_err, color=typecolors[abstype], 
                        alpha=0.3, step='mid', linestyle='')
        dv_abs = row[f'DV_{abstype}_ABS']
        if not np.isnan(dv_abs):
            dv_abs_shifted = dv_abs + row['DELTAV_LYA']  # Shift absorber velocity to systemic
            ax.axvline(dv_abs_shifted, color='red', linestyle='--', alpha=0.7)
        
        # Only show x-tick labels on the last (bottom) absorption subplot
        if is_last:
            ax.set_xlabel('Velocity relative to systemic (km/s)')
            ax.tick_params(labelbottom=True)
        else:
            ax.tick_params(labelbottom=False)

        ax.legend(loc='lower right', frameon=True, facecolor='white', edgecolor='none', framealpha=0.8)

# plt.suptitle("Lyman Alpha Sources with Absorbers and Systemic Velocities", fontsize=16)
plt.savefig(f"../plots/absorption_profiles_{SPEC_TYPE}.pdf", 
            dpi=500, bbox_inches='tight')
plt.show()
plt.close(fig)
